# 02 — Data Preparation

This notebook generates and audits the reproducible image dataset used by every model. It runs `src/data/process_data.py`, which validates images, removes exact duplicates from the usable pool, assigns a **grouped and stratified train / validation / test split** (groups connected by `product_code` or pixel hash never cross partitions) and computes class weights from the training partition only.

It also defines the shared PyTorch input pipeline — `Dataset`, `DataLoader`, resizing, normalisation and augmentation — so that all architectures (Dense, CNN, LSTM, Transformer, pretrained) are compared on exactly the same data.

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

## Objectives

- Generate or reuse the validated publication referenced by `data/processed/CURRENT.json`.
- Verify checksums, split sizes, class distribution per split and exclusion reasons.
- Check that no `product_code` or duplicated image crosses partitions (leakage control).
- Build a reusable `Dataset`/`DataLoader` that reads the manifest and returns `(image_tensor, label)`.
- Fix the preprocessing configuration: input resolution, channel normalisation statistics computed on training data, and the augmentation policy.
- Check the available hardware (Colab GPU, VRAM) and estimate loading throughput.

## Preview of the work

1. **Generate or reuse the publication.** run the pipeline with the class set chosen in notebook 01; never use a PREVIEW run.
2. **Audit the manifest.** split counts, class balance, excluded images and `training_class_weights.json`.
3. **Leakage checks.** groups by `product_code` and pixel hash are confined to one partition.
4. **Image preprocessing.** resize policy (e.g. 64×64 for Dense/LSTM, 128–224 for CNN/ViT), padding vs. cropping, normalisation.
5. **Data augmentation.** flips, small rotations, colour jitter; visual check that augmentations keep the class meaning.
6. **PyTorch `Dataset` and `DataLoader`.** implementation in `src/data/`, batch shapes and loading speed.
7. **Hardware and reproducibility.** seeds, deterministic flags, `nvidia-smi` and the configuration saved with every experiment.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Decisions from notebook 01, `data/raw/` |
| **Outputs** | `data/processed/runs/<run>/` (manifest, labels, class weights, provenance), reusable loaders in `src/data/` |
| **Course material** | `pytorch_dataset_dataloader-v2.pptx`, `MLLB_Colab_GPUs.pptx`, `Pytorch_quickstart_tutorial_FashionMNIST.ipynb` |